<div style="display: flex; align-items: center;">
    <h1>Differentiable irrigation control with WOFOST</h1>
    <img src="https://raw.githubusercontent.com/WUR-AI/diffWOFOST/refs/heads/main/docs/logo/diffwofost.png" width="150" style="margin-left: 20px;">
</div>


**What this notebook demonstrates**

By the end of the notebook we will have:

- a process-based crop model with a differentiable water balance;
- a neural-network irrigation policy;
- gradients propagated from economic return through WOFOST to the policy;
- a closed-loop irrigation strategy optimized with Adam through WOFOST;
- a policy that can be frozen and evaluated under different weather.

The goal is not to find the optimal irrigation strategy for Spanish sugar beet, but to demonstrate how a differentiable process model can be used as part of a control-optimization loop.


## 1. What is differentiable control?

A crop model can be used not only to predict what will happen to a crop, but also to decide what to do.

Consider irrigation. At each point in the season, a farmer can decide whether to apply water. That decision changes soil moisture, which changes crop water stress, which affects crop growth and ultimately yield.

This creates a feedback loop:

observe the crop → decide → simulate the consequence → observe the new state → decide again.

This is a control problem.

**Prediction versus control**

In ordinary supervised learning, the model predicts an outcome:

$$
x \rightarrow \hat{y}.
$$

In control, the model chooses an action that changes the future state:

$$
x_t \rightarrow a_t \rightarrow x_{t+1} \rightarrow x_{t+2} \rightarrow \cdots
$$

The action therefore affects the data that the controller will see later.

Irrigation is a particularly intuitive example: applying water today changes tomorrow's soil moisture, which can change whether irrigation is needed tomorrow.

Irrigation is therefore a closed-loop control problem: at each decision point, the policy observes the current state and chooses an action. The action changes the future crop and soil state, which in turn affects subsequent decisions. This differs from an open-loop schedule, where the irrigation dates would be fixed in advance.

In this notebook, the policy is a small neural network. The network observes the current state of the crop and soil and outputs an irrigation action. WOFOST then simulates the consequences of that action. The information available to the controller includes the current soil and crop state, current weather, and the time of year. Over a growing season, the sequence of decisions produces a crop trajectory and ultimately a harvest yield.

As equations,

$$
a_t = \pi_\omega(x_t),
\qquad
x_{t+1} = f_\theta(x_t, w_t, a_t),
$$

where $\omega$ are the network weights and $\theta$ the crop-model parameters.

**Why does differentiability matter?**

Suppose the controller irrigates today and the crop produces 100 kg/ha more beet at harvest.

In a conventional simulation, we could observe that outcome by running the model again with a different irrigation decision. But we would not automatically know how to change the neural-network parameters that produced that decision.

In a differentiable model, we can propagate this information backwards through the simulation. The model tells us how a small change in the policy parameters would affect the final objective.

This is what makes gradient-based control possible.

This is different from supervised learning. We do not have a dataset containing the correct irrigation decision for each day. Instead, the crop model itself provides the consequences of the decisions, and the economic objective provides the signal used to optimize the policy.

**Where does the gradient come from?**

The key idea is simple: if we can differentiate through both the irrigation policy and WOFOST, we can ask how changing the policy parameters would change the final economic return.

In other words, we can trace the effect of a small change in the neural network all the way through:

policy → irrigation → soil water → crop stress → growth → yield → economic return.

This gives us a gradient that tells us how to change the policy:

$$
\frac{\partial J}{\partial \omega}.
$$

That gradient is *not* learned from historical examples of irrigation and yield. It is obtained by differentiating through the simulated season, so standard optimizers such as Adam can improve $\omega$ directly.

In this example the seasonal objective $J$ combines sugar-beet revenue with the cost of irrigation events and millimetres. The controller therefore has to decide when applying water is worth the bill — which is the control problem the rest of the notebook sets up and solves.


### Software requirements

Install the latest `diffwofost` if needed. The notebook also uses `pandas`, `matplotlib` and `tqdm`.
Crop, soil and 2010 weather come from PCSE's water-limited sugar-beet YAML
(`test_waterlimitedproduction_wofost72_03.yaml`: 39.30°N, 3.43°W, 629 m).
Held-out years use NASA POWER at the same site.

**Code visibility.** In Google Colab, cells marked *Implementation details* are collapsed (click the title to expand). Visible code is the conceptual core: the essential STE policy sketch, the gradient probe, and the Adam idea. Plumbing, the full PCSE-aware MLP, plotting and checkpointing stay hidden by default.


In [ ]:
#@title Install packages { display-mode: "form" }
%%capture
# install required packages when needed
!pip install -q diffwofost matplotlib pandas tqdm


In [ ]:
%matplotlib inline

from pathlib import Path
import copy
import datetime as dt
import importlib.util
import urllib.request
import warnings

import matplotlib
matplotlib.style.use("ggplot")
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm.auto import tqdm
from pcse.input import NASAPowerWeatherDataProvider
from pcse.exceptions import PCSEError

from diffwofost.physical_models.config import ComputeConfig, Configuration
from diffwofost.physical_models.crop.wofost72 import Wofost72
from diffwofost.physical_models.engine import Engine
from diffwofost.physical_models.soil.classic_waterbalance import WaterbalanceFD, WaterbalancePP

if importlib.util.find_spec("diffwofost.physical_models.test") is not None:
    from diffwofost.physical_models.test import get_test_data, prepare_engine_input
else:
    # PyPI 0.5.0 still exports these from utils.
    from diffwofost.physical_models.utils import get_test_data, prepare_engine_input

warnings.filterwarnings("ignore", message="To copy construct from a tensor.*")
ComputeConfig.set_device("cpu")
ComputeConfig.set_dtype(torch.float64)

print(f"torch version: {torch.__version__}")
print(f"device: {ComputeConfig.get_device()}")
print(f"dtype: {ComputeConfig.get_dtype()}")


## 2. The crop-model environment

The plant is PCSE's water-limited WOFOST 7.2 case `test_waterlimitedproduction_wofost72_03.yaml`: sugar beet sown on 27 March 2010 at **39.30°N, 3.43°W**, elevation 629 m (Castilla-La Mancha), with the YAML soil (`SMW` / `SMFCF` / `SM0` ≈ 0.15 / 0.32 / 0.42). The freely draining water balance (`WaterbalanceFD`) has no capillary rise from groundwater.

Two runs bound what water can do:

- **Rainfed**: no irrigation.
- **Potential production**: soil moisture held at field capacity. This is the physiological yield ceiling if water were never limiting — not an irrigated operating plan and not an economic score.

WOFOST reports `TWSO` as kg/ha of *dry* storage-organ biomass. Fresh beet yield uses a dry-matter fraction of 0.23:

$$
Y_{\mathrm{fresh}} = \frac{\mathrm{TWSO}}{1000 \times 0.23}\quad[\mathrm{t/ha}].
$$


In [ ]:
#@title Load crop, soil, and 2010 weather { display-mode: "form" }
def download_if_needed(path, url):
    path = Path(path)
    if path.exists():
        print(f"Using local file: {path}")
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path.name}")
    return path


filename = "test_waterlimitedproduction_wofost72_03.yaml"
test_data_path = download_if_needed(
    filename,
    "https://raw.githubusercontent.com/ajwdewit/pcse/refs/heads/master/"
    f"tests/test_data/{filename}",
)
test_data = get_test_data(test_data_path)

crop_model_params = [
    "SPAN", "TDWI", "TBASE", "PERDL", "RGRLAI", "KDIFTB", "SLATB",
    "TSUMEM", "TBASEM", "TEFFMX", "TSUM1", "TSUM2", "DLO", "DLC", "DVSI", "DVSEND", "DTSMTB",
    "AMAXTB", "EFFTB", "TMPFTB", "TMNFTB",
    "Q10", "RMR", "RML", "RMS", "RMO", "RFSETB",
    "CFET", "DEPNR", "IAIRDU", "IOX", "CRAIRC", "SM0", "SMW", "SMFCF", "WAV",
    "RDI", "RRI", "RDMCR", "RDMSOL", "RDRRTB",
    "RDRSTB", "SSATB", "SPA",
    "FRTB", "FLTB", "FSTB", "FOTB",
    "CVL", "CVO", "CVR", "CVS",
    "SOPE", "KSUB", "SMLIM",
]
provider, weather, yaml_agro, _ = prepare_engine_input(test_data, crop_model_params)
agromanagement = yaml_agro

TRAIN_YEAR = 2010
TEST_YEARS = (2009, 2011)


def _as_date(day):
    if isinstance(day, dt.datetime):
        return day.date()
    return day


def drv_get(drv, name):
    if isinstance(drv, dict):
        return drv[name]
    return getattr(drv, name)


def weather_is_pcse_provider(weather):
    return callable(weather) and not isinstance(weather, (list, tuple))


def weather_scalar(row, name):
    value = row[name] if isinstance(row, dict) else getattr(row, name)
    if isinstance(value, torch.Tensor):
        return float(value.detach())
    return float(value)


def weather_on_day(weather, day):
    day = _as_date(day)
    if weather_is_pcse_provider(weather):
        return weather(day)
    for row in weather:
        row_day = row["DAY"] if isinstance(row, dict) else getattr(row, "DAY")
        if _as_date(row_day) == day:
            return row
    raise KeyError(day)


def rain_mm_on(weather, day):
    return 10.0 * weather_scalar(weather_on_day(weather, day), "RAIN")


def pcse_provider_to_engine_weather(provider, start, end):
    """Daily dicts for Engine.setup on current main; unused on PyPI 0.5.0."""
    dtype = ComputeConfig.get_dtype()
    device = ComputeConfig.get_device()
    names = [
        "LAT", "LON", "ELEV", "IRRAD", "TMIN", "TMAX", "VAP", "RAIN",
        "E0", "ES0", "ET0", "WIND", "TEMP",
    ]
    rows = []
    day = start
    one = dt.timedelta(days=1)
    while day <= end:
        w = provider(day)
        row = {"DAY": day}
        for name in names:
            if hasattr(w, name):
                row[name] = torch.as_tensor(float(getattr(w, name)), dtype=dtype, device=device)
        if "DTEMP" not in row and "TEMP" in row and "TMAX" in row:
            row["DTEMP"] = 0.5 * (row["TEMP"] + row["TMAX"])
        rows.append(row)
        day += one
    return rows


campaign_start = next(iter(agromanagement[0].keys()))
site_row = weather[0] if isinstance(weather, list) else weather(campaign_start)
SITE_LAT = weather_scalar(site_row, "LAT")
SITE_LON = weather_scalar(site_row, "LON")
SITE_ELEV = weather_scalar(site_row, "ELEV")


def agro_for_year(agromanagement, year):
    shifted = []
    for campaign in copy.deepcopy(agromanagement):
        new_campaign = {}
        for start, spec in campaign.items():
            calendar = spec["CropCalendar"]
            for key in ("crop_start_date", "crop_end_date"):
                calendar[key] = calendar[key].replace(year=year)
            new_campaign[start.replace(year=year)] = spec
        shifted.append(new_campaign)
    return shifted


class NASAPowerWindow(NASAPowerWeatherDataProvider):
    """NASA POWER with a finite past window.

    PCSE's default query uses ``user=anonymous`` and ``end=today``, which the
    POWER API currently rejects with HTTP 422.
    """

    def __init__(self, latitude, longitude, start, end, **kwargs):
        self._window_start = start
        self._window_end = end
        super().__init__(latitude, longitude, **kwargs)

    def _query_NASAPower_server(self, latitude, longitude):
        import requests

        server = "https://power.larc.nasa.gov/api/temporal/daily/point"
        payload = {
            "request": "execute",
            "parameters": ",".join(self.power_variables),
            "latitude": latitude,
            "longitude": longitude,
            "start": self._window_start.strftime("%Y%m%d"),
            "end": self._window_end.strftime("%Y%m%d"),
            "community": "AG",
            "format": "JSON",
        }
        req = requests.get(server, params=payload)
        if req.status_code != self.HTTP_OK:
            raise PCSEError(
                f"Failed retrieving POWER data, server returned HTTP {req.status_code} "
                f"on {req.url}"
            )
        return req.json()


print(f"crop: {test_data['ModelParameters'].get('CRPNAM', 'unknown')}")
print("soil: YAML (original test file)")
print(
    f"weather: YAML {TRAIN_YEAR}  {SITE_LAT:.5f}°N, {abs(SITE_LON):.5f}°W  "
    f"elev {SITE_ELEV:.0f} m  (Castilla-La Mancha)"
)
print(f"campaign year: {TRAIN_YEAR} (train); NASA POWER held-out years {TEST_YEARS}")
print(f"WAV (initial available water): {float(provider['WAV']):.1f} cm")
print(
    "soil moisture bounds SMW / SMFCF / SM0: "
    f"{float(provider['SMW']):.3f} / {float(provider['SMFCF']):.3f} / {float(provider['SM0']):.3f}"
)


In [ ]:
#@title Rainfed and potential production runs { display-mode: "form" }
wlp_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalanceFD,
    OUTPUT_VARS=["DVS", "LAI", "SM", "TAGP", "TRA", "RFTRA", "TWSO", "TWST", "RD"],
)

pp_config = Configuration(
    CROP=Wofost72,
    SOIL=WaterbalancePP,
    OUTPUT_VARS=["DVS", "LAI", "SM", "TAGP", "RFTRA", "TWSO"],
)


def scalarize(value):
    if isinstance(value, torch.Tensor):
        return float(value.detach().cpu())
    return value


def results_to_frame(results, irrigation=None):
    rows = []
    for row in results:
        rows.append({key: (value if key == "day" else scalarize(value)) for key, value in row.items()})
    frame = pd.DataFrame(rows)
    frame["day"] = pd.to_datetime(frame["day"])
    frame = frame.set_index("day")
    if irrigation is not None:
        irr = pd.Series(
            {day: 10.0 * scalarize(amount) for day, amount in irrigation},
            name="irrigation",
        )
        irr.index = pd.to_datetime(irr.index)
        frame = frame.join(irr, how="left")
        frame["irrigation"] = frame["irrigation"].fillna(0.0)
    return frame


def summarize(name, frame, totirr=0.0):
    emerged = frame[frame["DVS"] > 0]
    n_irr = int((frame.get("irrigation", pd.Series(0, index=frame.index)) > 15.0).sum())
    print(
        f"{name:22s}  TWSO={frame['TWSO'].iloc[-1]:8.1f} kg/ha  "
        f"TOTIRR={totirr:5.1f} cm  min RFTRA={emerged['RFTRA'].min():.2f}  "
        f"events>15 mm={n_irr:3d}"
    )


rainfed_engine = Engine(config=wlp_config)
rainfed_engine.setup(provider, weather, agromanagement)
rainfed_engine.run_till_terminate()
rainfed_df = results_to_frame(rainfed_engine.get_output())

pp_engine = Engine(config=pp_config)
pp_engine.setup(provider, weather, agromanagement)
pp_engine.run_till_terminate()
pp_df = results_to_frame(pp_engine.get_output())

summarize("rainfed", rainfed_df, totirr=0.0)
summarize("potential production", pp_df, totirr=float("nan"))
print(
    "yield gap due to water: "
    f"{pp_df['TWSO'].iloc[-1] - rainfed_df['TWSO'].iloc[-1]:.0f} kg/ha dry  "
    f"({(pp_df['TWSO'].iloc[-1] - rainfed_df['TWSO'].iloc[-1]) / 1000 / 0.23:.1f} t/ha fresh at 23% DM)"
)


The rainfed crop loses a large fraction of potential storage-organ biomass. Soil moisture falls well below field capacity during canopy expansion, `RFTRA` drops, and growth of the beet (`TWSO`) stalls.

This is exactly the situation in which control is interesting. There is substantial value in applying water, but water is not free. Applying water too early wastes part of its benefit; applying it too late can allow irreversible yield loss. The controller therefore has to decide *when* water is worth applying.

This gives us a useful control problem: there is a large potential benefit from irrigation, but we do not want to apply water unnecessarily because irrigation itself has a cost.


In [ ]:
#@title Plot rainfed vs potential (SM, RFTRA, TWSO) { display-mode: "form" }
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

axes[0].plot(rainfed_df.index, rainfed_df["SM"], label="rainfed")
axes[0].plot(pp_df.index, pp_df["SM"], label="potential", linestyle="--")
axes[0].axhline(float(provider["SMFCF"]), color="0.4", linewidth=0.8, linestyle=":")
axes[0].axhline(float(provider["SMW"]), color="0.4", linewidth=0.8, linestyle=":")
axes[0].set_ylabel("SM $(-)$")
axes[0].legend()

axes[1].plot(rainfed_df.index, rainfed_df["RFTRA"], label="rainfed")
axes[1].plot(pp_df.index, pp_df["RFTRA"], label="potential", linestyle="--")
axes[1].set_ylabel("RFTRA $(-)$")

axes[2].plot(rainfed_df.index, rainfed_df["TWSO"], label="rainfed")
axes[2].plot(pp_df.index, pp_df["TWSO"], label="potential", linestyle="--")
axes[2].set_ylabel("TWSO (kg ha$^{-1}$)")
axes[2].set_xlabel("day")

fig.suptitle("Water-limited rainfed crop versus potential production", y=0.99)
fig.tight_layout()
plt.show()


## 3. Reference irrigation strategies

**Why include AIMCRA?**

AIMCRA provides an agronomic benchmark against which we can interpret the learned controller. It is not used to train the neural network.

This distinction is important: the goal of this notebook is not to reproduce an existing irrigation rule. We want to see whether a neural policy can discover useful irrigation behaviour by optimizing the economic objective through WOFOST.

We therefore compare three things:

- **Rainfed**: what happens if we never irrigate?
- **AIMCRA**: what does an established agronomic rule do?
- **Learned policy**: what does gradient-based optimization discover?

**Potential production** remains the water-unlimited yield ceiling (not scored economically).

Spain's [*Técnicas de riego en la remolacha azucarera*](https://www.aimcra.es/publicaciones/documentos/otras/tecnicas_de_riego.pdf) (2001) irrigates from a water-balance / NAP rule: when about 35 mm has gone from the current root zone, apply a 40 mm *net* gift (50 mm applied at 80% efficiency), skip a useful rain day, after emergence, last irrigation 15 September. That is the AIMCRA handbook rule we re-implement as a closed-loop reference — not the policy class we train.


## 4. Defining the control problem

On a decision day the controller chooses either:

- **OFF**: no irrigation
- **ON**: apply a pulse of up to 50 mm, limited by how much the root zone can currently store

$$
I_t = \min\bigl(50\,\mathrm{mm},\; \text{refill to field capacity}\bigr)
\quad\text{or}\quad 0.
$$

For this demonstration, we model irrigation as an event-based operation, not a continuous pump. That AIMCRA-sized pulse is the action; the refill clip prevents overfilling an already wet root zone.

**A coarse training grid.** To keep the differentiable simulation relatively small, we train the controller with one decision every seven days. The crop model itself still evolves daily.

After training, however, the learned policy is not a calendar of weekly decisions. It is a function

$$
a_t = \pi_\omega(x_t).
$$

We can therefore evaluate the same function every day. The policy simply receives the current state and makes a new decision.

This is an important distinction between a policy and a schedule: the learned object is the mapping from state to action, not the sequence of dates encountered during training.

After a 50 mm gift the next-day refill cap is small, so neighbouring days usually stay OFF even when the same weights are allowed to act every day — daily deployment need not turn into a flood.

The policy writes its action into the water balance before the daily rates are calculated, so the gradient can flow from harvest yield back through soil water into $\omega$. Unit conversions and PCSE bookkeeping are in the collapsed implementation cells.


## 5. A neural irrigation policy

The controller is a small MLP. It does not store a calendar. It maps the current observation to the probability of an event.

| Input | Meaning | What the controller can infer |
| --- | --- | --- |
| `cap` | water required to refill the root zone | “How much water is missing?” |
| rain | today's rainfall | “Will rain provide water instead?” |
| DVS | crop development stage | “How important is water at this stage?” |
| DOY | day of year | “Where are we in the season?” |

These are deliberately simple observations. The policy does not receive WOFOST's internal `RFTRA` state or the future weather sequence — the same information class as a water-balance rule.

$$
x_t = \bigl(\mathrm{cap}/50,\; \mathrm{rain},\; \mathrm{DVS}/2,\; \mathrm{DOY}/365\bigr),
\qquad
p_t = \sigma\bigl(\mathrm{MLP}_\omega(x_t)\bigr),
\qquad
a_t = \pi_\omega(x_t).
$$

$$
a_t = \pi_\omega(x_t)
\quad\text{feeds}\quad
x_{t+1} = f_\theta(x_t, w_t, a_t).
$$


## 6. Making discrete control differentiable

A problem now appears. The farmer's decision is discrete:

$$
a_t \in \{0, 1\}.
$$

A hard threshold such as

$$
a_t = \mathbf{1}[p_t > 0.5]
$$

is not differentiable. If we changed the probability from 0.49 to 0.51, the action suddenly jumps from OFF to ON — and that jump has no useful gradient for Adam.

We therefore use a **straight-through estimator (STE)**. The forward pass uses the real discrete decision; the backward pass treats the gate as if it were the smooth sigmoid, so the gradient of $p_t$ can flow into $\omega$. Think of this as using a realistic discrete decision in the forward simulation while giving the optimizer a smooth approximation of how that decision could change.

$$
a_t =
\begin{cases}
I_t & p_t > 0.5 \\
0 & p_t \le 0.5
\end{cases}
\quad\text{(forward)},
\qquad
\frac{\partial a_t}{\partial p_t} \;\approx\;
\frac{\partial\,\sigma}{\partial p}\quad\text{(backward)}.
$$

```
              forward
p ───────────→ hard ON/OFF ─────→ WOFOST
│
└─────────────→ surrogate gradient
              backward
```

The optimizer sees the differentiable surrogate, but the farmer ultimately uses the discrete policy. These are therefore not necessarily optimized equally well at every iteration. We consequently evaluate the actual hard policy after each update and retain its best checkpoint — a useful general lesson about surrogate-gradient optimization.


In code, the control loop is essentially:

```python
state = observe_crop_and_soil()
p = policy(state)
irrigation = make_differentiable_decision(p)   # STE hard gate

wofost.step(irrigation)
return_ = economic_return(wofost)
loss = -return_

loss.backward()
optimizer.step()
```

The cells below keep a short essential sketch of the policy visible. PCSE integration, the full `IrrigationMLP`, AIMCRA, economics helpers and evaluation utilities are collapsed as implementation details.


In [ ]:
#@title Implementation details — helpers, AIMCRA, WOFOST loop, economics { display-mode: "form" }
IRRIGATION_EFFICIENCY = 0.8
DM_FRACTION = 0.23
BEET_PRICE = 40.0
C_MM = 1.0
C_EVENT = 25.0
EVENT_MM_MIN = 1.0
EVENT_TEMP_MM = 0.1
MLP_DOSE_MM = 50.0
MLP_HIDDEN = 64
MLP_N_OBS = 4
DECISION_PERIOD = 7  # training grid
DEPLOY_PERIOD = 1    # same weights, every day


def kiosk_get(engine, name, default):
    if name in engine.kiosk:
        value = engine.kiosk[name]
        if value is not None:
            return value
    return default


def as_scalar(value, like):
    tensor = torch.as_tensor(value, dtype=like.dtype, device=like.device)
    return tensor.reshape(()) if tensor.numel() == 1 else tensor


def stacked_amounts(irrigation):
    return torch.stack([amount for _, amount in irrigation])


def fresh_yield_t_ha(twso):
    return twso / 1000.0 / DM_FRACTION


def irrigation_economics(twso, amounts_cm, n_events=None, c_event=None, c_mm=None):
    """R = 40 Y - 25 N - I. N is any day with applied water."""
    y_fresh = fresh_yield_t_ha(twso)
    revenue = BEET_PRICE * y_fresh
    applied_mm = amounts_cm * 10.0
    if n_events is None:
        n_events = torch.sigmoid((applied_mm - EVENT_MM_MIN) / EVENT_TEMP_MM).sum()
    c_event = C_EVENT if c_event is None else c_event
    c_mm = C_MM if c_mm is None else c_mm
    cost = c_event * n_events + c_mm * applied_mm.sum()
    reward = revenue - cost
    n_hard = int((applied_mm.detach() > EVENT_MM_MIN).sum())
    return {
        "y_fresh": y_fresh,
        "revenue": revenue,
        "cost": cost,
        "reward": reward,
        "applied_mm": applied_mm.sum(),
        "n_events": n_events,
        "n_events_hard": n_hard,
    }


def is_decision_day(doy, period=DECISION_PERIOD):
    if period <= 1:
        return True
    return ((int(doy) - 1) % int(period)) == 0


class StandardPracticePolicy(torch.nn.Module):
    """AIMCRA water-balance / NAP rule. Not trained."""

    def __init__(
        self,
        net_dose_mm=40.0,
        nap_mm=35.0,
        nap_frac=0.65,
        useful_rain_mm=10.0,
        last_month=9,
        last_day=15,
        efficiency=None,
    ):
        super().__init__()
        self.net_dose_mm = net_dose_mm
        self.nap_mm = nap_mm
        self.nap_frac = nap_frac
        self.useful_rain_mm = useful_rain_mm
        self.last_month = last_month
        self.last_day = last_day
        self.efficiency = IRRIGATION_EFFICIENCY if efficiency is None else efficiency

    def reset(self):
        return None

    def forward(self, engine):
        sm = engine.soil.states.SM
        zero = torch.zeros((), dtype=sm.dtype, device=sm.device)
        day = engine.day
        if (day.month, day.day) > (self.last_month, self.last_day):
            return zero
        rain_mm = float(as_scalar(drv_get(engine.drv, "RAIN"), sm).detach()) * 10.0
        dvs = float(as_scalar(kiosk_get(engine, "DVS", zero), sm).detach())
        if dvs <= 0.0 or rain_mm >= self.useful_rain_mm:
            return zero
        smw = engine.soil.params.SMW
        smfc = engine.soil.params.SMFCF
        rd = as_scalar(
            kiosk_get(engine, "RD", torch.tensor(10.0, dtype=sm.dtype, device=sm.device)),
            sm,
        ).clamp_min(1.0)
        rd_cm = float(rd.detach())
        available_mm = max(float((smfc - smw).detach()) * rd_cm * 10.0, 1.0)
        nap_mm = min(self.nap_mm, self.nap_frac * available_mm)
        depletion_mm = float((smfc - sm).clamp(min=0.0).detach()) * rd_cm * 10.0
        if depletion_mm < nap_mm:
            return zero
        deficit_cm = torch.clamp((smfc - sm) * rd, min=0.0)
        applied_cap = deficit_cm / self.efficiency
        target_cm = self.net_dose_mm / 10.0 / self.efficiency
        applied_cm = torch.minimum(
            applied_cap, torch.as_tensor(target_cm, dtype=sm.dtype, device=sm.device)
        )
        if float(applied_cm.detach()) * 10.0 < EVENT_MM_MIN:
            return zero
        return applied_cm


def run_with_policy(engine, policy=None, efficiency=IRRIGATION_EFFICIENCY):
    """PCSE daily loop; the policy writes a_t (cm) into the water balance."""
    zero = torch.zeros((), dtype=ComputeConfig.get_dtype(), device=ComputeConfig.get_device())
    irrigation = []
    while engine.flag_terminate is False:
        engine.day, delt = engine.timer()
        engine.integrate(engine.day, delt)
        engine.drv = engine._get_driving_variables(engine.day)
        engine.agromanager(engine.day, engine.drv)
        amount_cm = zero if policy is None else policy(engine)
        engine.soil._RIRR = amount_cm * efficiency
        irrigation.append((engine.day, amount_cm))
        engine.calc_rates(engine.day, engine.drv)
        if engine.flag_terminate is True:
            engine._terminate_simulation(engine.day)
    return irrigation


def evaluate_policy(policy, provider, weather, agromanagement, config, c_event=None):
    if policy is not None and hasattr(policy, "reset"):
        policy.reset()
    engine = Engine(config=config)
    engine.setup(provider, weather, agromanagement)
    irrigation = run_with_policy(engine, policy=policy)
    results = engine.get_output()
    amounts_cm = stacked_amounts(irrigation)
    twso = results[-1]["TWSO"]
    totirr = engine.soil.states.TOTIRR
    n_events = None
    if policy is not None and getattr(policy, "event_terms", None):
        n_events = torch.stack(policy.event_terms).sum()
    eco = irrigation_economics(twso, amounts_cm, n_events=n_events, c_event=c_event)
    return {
        "engine": engine,
        "results": results,
        "frame": results_to_frame(results, irrigation),
        "irrigation": irrigation,
        "amounts_cm": amounts_cm,
        "twso": twso,
        "totirr": totirr,
        **eco,
    }


**The essential policy.** The network maps the current observation to an irrigation probability, then uses an STE hard gate:

```python
x = policy_inputs(state)          # (cap/50, rain, DVS/2, DOY/365)
p = torch.sigmoid(self.network(x))

hard = (p > 0.5).float()
gate = hard.detach() + p - p.detach()   # STE: hard forward, soft backward

return gate * dose                  # dose ≤ 50 mm, clipped to refill capacity
```

That is the control map $x_t \rightarrow a_t$. The collapsed cell below is the full PCSE-aware implementation used in the rest of the notebook.


In [ ]:
#@title Implementation details — PCSE state access, logging, clipping and bookkeeping { display-mode: "form" }
def policy_inputs(engine):
    """Observation x_t: refill cap, rain, DVS, day of year, plus depletion in mm."""
    sm = engine.soil.states.SM
    smfc = engine.soil.params.SMFCF
    rain = as_scalar(drv_get(engine.drv, "RAIN"), sm)
    dvs = as_scalar(kiosk_get(engine, "DVS", torch.zeros_like(sm)), sm)
    rd = as_scalar(
        kiosk_get(engine, "RD", torch.tensor(10.0, dtype=sm.dtype, device=sm.device)),
        sm,
    ).clamp_min(1.0)
    doy = torch.as_tensor(float(engine.day.timetuple().tm_yday), dtype=sm.dtype, device=sm.device)
    deficit_cm = torch.clamp((smfc - sm) * rd, min=0.0)
    applied_cap_mm = deficit_cm / IRRIGATION_EFFICIENCY * 10.0
    depletion_mm = deficit_cm * 10.0
    x = torch.stack([applied_cap_mm / 50.0, rain, dvs / 2.0, doy / 365.0])
    return x, applied_cap_mm, dvs, depletion_mm, rain


class IrrigationMLP(torch.nn.Module):
    """Closed-loop event policy: p = σ(MLP(x)), hard ON if p > 0.5 (STE).

    Train with period=7; deploy with period=1.
    """

    def __init__(self, hidden=MLP_HIDDEN, dose_mm=MLP_DOSE_MM, period=DECISION_PERIOD):
        super().__init__()
        dtype = ComputeConfig.get_dtype()
        device = ComputeConfig.get_device()
        self.dose_cm = dose_mm / 10.0
        self.period = int(period)
        self.hard_decisions = False
        self.body = torch.nn.Sequential(
            torch.nn.Linear(MLP_N_OBS, hidden, dtype=dtype, device=device),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden, dtype=dtype, device=device),
            torch.nn.Tanh(),
        )
        self.actor = torch.nn.Linear(hidden, 1, dtype=dtype, device=device)
        self.event_terms = []
        self.p_terms = []
        self.records = []

    def reset(self):
        self.event_terms = []
        self.p_terms = []
        self.records = []

    def forward(self, engine):
        sm = engine.soil.states.SM
        zero = torch.zeros((), dtype=sm.dtype, device=sm.device)
        if not is_decision_day(engine.day.timetuple().tm_yday, self.period):
            return zero
        x, cap_mm, dvs, depletion_mm, rain = policy_inputs(engine)
        logit = self.actor(self.body(x)).reshape(())
        p = torch.sigmoid(logit)
        crop = (dvs > 0).to(dtype=sm.dtype)
        if self.hard_decisions:
            hard = (p > 0.5).to(dtype=p.dtype)
            gate = crop * (hard.detach() + p - p.detach())
        else:
            gate = crop * p
        dose = torch.as_tensor(self.dose_cm, dtype=sm.dtype, device=sm.device)
        applied = torch.minimum(dose, cap_mm / 10.0)
        gate = gate * (1.0 - (applied * 10.0 < EVENT_MM_MIN).to(dtype=sm.dtype))
        self.event_terms.append(gate)
        self.p_terms.append(crop * p)
        self.records.append(
            {
                "day": engine.day,
                "p": p.detach(),
                "depletion_mm": depletion_mm.detach(),
                "rain_mm": (rain * 10.0).detach(),
                "dvs": dvs.detach(),
                "applied_mm": (gate * applied * 10.0).detach(),
            }
        )
        return gate * applied


In [ ]:
#@title Implementation details — reference economic summaries { display-mode: "form" }
def summarize_economics(name, outcome=None, frame=None, amounts_cm=None, potential=False):
    if frame is None:
        frame = outcome["frame"]
    twso = frame["TWSO"].iloc[-1] if outcome is None else outcome["twso"]
    if potential:
        y = scalarize(fresh_yield_t_ha(twso))
        print(f"{name:28s}  fresh={y:5.1f} t/ha  (yield ceiling; not scored economically)")
        return
    if outcome is not None:
        eco = outcome
        amounts = outcome["amounts_cm"]
    else:
        amounts = amounts_cm
        eco = irrigation_economics(torch.as_tensor(twso, dtype=ComputeConfig.get_dtype()), amounts)
    y = scalarize(eco["y_fresh"])
    mm = scalarize(eco["applied_mm"])
    n_hard = eco["n_events_hard"] if outcome is not None else int((amounts.detach() * 10 > EVENT_MM_MIN).sum())
    print(
        f"{name:28s}  fresh={y:5.1f} t/ha  applied={mm:5.0f} mm  events={n_hard:3d}  "
        f"revenue={scalarize(eco['revenue']):7.0f} €/ha  "
        f"cost={scalarize(eco['cost']):5.0f}  reward={scalarize(eco['reward']):7.0f} €/ha"
    )


zero_amounts = torch.zeros(len(rainfed_df), dtype=ComputeConfig.get_dtype())
summarize_economics("rainfed", frame=rainfed_df, amounts_cm=zero_amounts)
summarize_economics("potential production", frame=pp_df, potential=True)

standard_practice = evaluate_policy(
    StandardPracticePolicy(), provider, weather, agromanagement, wlp_config
)
summarize_economics("AIMCRA (reference)", standard_practice)


## 7. The economic objective

The controller is not rewarded simply for producing more beet. It is rewarded for producing more *valuable* beet than the irrigation costs.

An irrigation event therefore has two competing effects:

- **Benefit**: more soil water → less crop stress → potentially higher yield.
- **Cost**: water volume plus the cost of operating the irrigation system.

The optimal policy must balance these effects. Mobilisation and volume are billed separately, so the seasonal return is

$$
R = 40\,Y_{\mathrm{fresh}} - 25\,N - I,
$$

with $Y_{\mathrm{fresh}}$ the fresh beet yield (t/ha), $N$ the number of irrigation events, and $I=\sum_t I_t$ the applied millimetres. Equivalently, each day's cost is

$$
C(I_t) =
\begin{cases}
0 & I_t = 0 \\
25 + I_t & I_t > 0.
\end{cases}
$$

| assumption | value |
|---|---|
| beet price | €40/t fresh |
| volume cost | €1.0 per mm per ha |
| event cost | €25/ha |
| application efficiency | 0.8 |
| dry-matter fraction | 0.23 |

Capital (borehole, pump, pipes, reel) is excluded. Potential production is kept as a physiological yield ceiling and is not given an economic score. Adam minimises $J=-R$. The code that evaluates $R$ lives in the collapsed implementation cell above.


A single STE episode is enough to see that $\omega$ receives a gradient through the water balance and the crop:

```
policy parameter → irrigation → soil water → crop growth → yield → R → ∇_ω R
```

**This is the central idea of differentiable control.**

We did not provide the network with examples of good irrigation decisions. Nevertheless, after one forward simulation, PyTorch can compute gradients for all of the policy parameters.

The gradient connects the economic outcome at harvest all the way back to the neural-network weights:

policy parameters → irrigation decisions → soil water → crop growth → yield → economic return.

Adam can now use this information to modify the policy. The norms printed below should be nonzero through the network layers: that is the signal traveling through WOFOST.


In [ ]:
#@title Gradient probe: ∂R/∂ω through WOFOST
torch.manual_seed(0)
probe = IrrigationMLP(period=DECISION_PERIOD)
probe.hard_decisions = True
probe_out = evaluate_policy(probe, provider, weather, agromanagement, wlp_config)

# The magic: differentiate the seasonal return through WOFOST into ω.
(-probe_out["reward"]).backward()

print(f"hard R = {scalarize(probe_out['reward']):.0f} €/ha   events = {probe_out['n_events_hard']}")
print("gradients through WOFOST:")
grad_sq = torch.zeros((), dtype=ComputeConfig.get_dtype())
for name, param in probe.named_parameters():
    if param.grad is None:
        print(f"  {name:20s}  (unused)")
        continue
    gnorm = float(param.grad.detach().norm())
    grad_sq = grad_sq + param.grad.detach().square().sum()
    print(f"  {name:20s}  ||∂R/∂ω|| piece = {gnorm:.3g}")
print(f"  total ||∂R/∂ω|| = {float(grad_sq.sqrt()):.3g}")


## 8. Optimizing the policy through WOFOST

**What does Adam do here?**

At this point we have everything needed for gradient-based optimization:

- the policy produces irrigation decisions;
- WOFOST simulates the season;
- the economic objective measures how good the season was;
- backpropagation tells us how the policy parameters affected that objective.

Adam repeatedly uses these gradients to update the policy parameters.

So the training loop is simply:

simulate → calculate return → backpropagate → update policy → simulate again.

Adam is asking, over and over: *how should I change $\omega$ so that the next simulated season is more profitable?* Because the gradient connects the final return to the network weights, it can do that without manually searching through irrigation calendars.

```
initialize π_ω
      ↓
forward through WOFOST   (weekly STE events)
      ↓
R = 40 Y − 25 N − I
      ↓
backpropagate  ∂J/∂ω , J = −R
      ↓
Adam update
      ↓
evaluate hard R; keep the best
      ↓
deploy the same ω every day
```

A short soft warmup lets $p$ move away from initialization; then the STE gate is the demonstration. After each update we score the deterministic hard policy — the one a farmer would actually run — and keep the checkpoint with the best hard $R$ on the weekly grid. After training, those best weights are read **every day**.


The Adam loop is conceptually:

```python
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

for step in range(n_steps):
    optimizer.zero_grad()
    outcome = evaluate_policy(policy, ...)
    loss = -outcome["reward"]
    loss.backward()
    optimizer.step()

    # The farmer uses the discrete policy — keep the best hard R.
    hard = evaluate_hard(policy)
```

The cell below is the full training machinery (warmup, STE phases, checkpointing, history). Expand it if you want the exact Colab-ready implementation.


In [ ]:
#@title Implementation details — training, checkpoints, and history { display-mode: "form" }
ckpt_dir = Path("data_temp")
ckpt_dir.mkdir(exist_ok=True)
CKPT = ckpt_dir / "mlp_weekly_best.pt"
CKPT_LEGACY = ckpt_dir / "mlp_aimcra_norftra_best.pt"


def clone_state(net):
    return {key: value.detach().clone() for key, value in net.state_dict().items()}


def hard_eval(net, weather_provider=weather, agro=agromanagement):
    was = net.hard_decisions
    net.hard_decisions = True
    with torch.no_grad():
        out = evaluate_policy(net, provider, weather_provider, agro, wlp_config)
    net.hard_decisions = was
    return out


def load_checkpoint(path):
    try:
        blob = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        blob = torch.load(path, map_location="cpu")
    if isinstance(blob, dict) and blob.get("best_weekly") is not None:
        state, hist, saved_r = blob["best_weekly"], blob.get("history", []), blob.get("best_w_r")
    elif isinstance(blob, dict) and "state_dict" in blob:
        state, hist, saved_r = blob["state_dict"], blob.get("history", []), blob.get("best_r")
    else:
        state, hist, saved_r = blob, [], None
    weight = state.get("body.0.weight") if isinstance(state, dict) else None
    in_dim = int(weight.shape[1]) if weight is not None else None
    if in_dim != MLP_N_OBS:
        return None
    return state, hist or [], saved_r


mlp = IrrigationMLP(period=DECISION_PERIOD)
history = []
best_state = None
best_hard_r = float("-inf")
loaded = False
for path in (CKPT, CKPT_LEGACY):
    if not path.exists():
        continue
    loaded_blob = load_checkpoint(path)
    if loaded_blob is None:
        continue
    state, history, saved_r = loaded_blob
    mlp.load_state_dict(state)
    best_state = clone_state(mlp)
    best_hard_r = float("-inf") if saved_r is None else float(saved_r)
    loaded = True
    print(f"loaded policy from {path}")
    break


def run_phase(n_steps, lr, use_hard, desc):
    global best_state, best_hard_r
    mlp.period = DECISION_PERIOD
    opt = torch.optim.Adam(mlp.parameters(), lr=lr)
    bar = tqdm(range(n_steps), desc=desc)
    for _ in bar:
        opt.zero_grad()
        mlp.hard_decisions = use_hard
        train_out = evaluate_policy(mlp, provider, weather, agromanagement, wlp_config)
        pre = clone_state(mlp)
        (-train_out["reward"]).backward()
        torch.nn.utils.clip_grad_norm_(mlp.parameters(), 0.5)
        opt.step()
        if use_hard:
            hard = train_out
            saved = pre
        else:
            hard = hard_eval(mlp)
            saved = clone_state(mlp)
        rec_r = scalarize(hard["reward"])
        if rec_r > best_hard_r:
            best_hard_r = rec_r
            best_state = saved
        history.append(
            {
                "step": len(history),
                "reward_hard": rec_r,
                "n_events": hard["n_events_hard"],
                "y_fresh": scalarize(hard["y_fresh"]),
            }
        )
        bar.set_postfix(
            R_hard=f"{rec_r:.0f}",
            best=f"{best_hard_r:.0f}",
            events=hard["n_events_hard"],
        )


if not loaded:
    torch.manual_seed(11)
    mlp = IrrigationMLP(period=DECISION_PERIOD)
    run_phase(40, 1e-3, use_hard=False, desc="soft warmup")
    run_phase(80, 1e-3, use_hard=True, desc="STE")
    if best_state is not None:
        mlp.load_state_dict(best_state)
    run_phase(120, 1e-4, use_hard=True, desc="STE (fine)")
    if best_state is not None:
        mlp.load_state_dict(best_state)
    torch.save({"state_dict": clone_state(mlp), "history": history, "best_r": best_hard_r}, CKPT)

if best_state is not None:
    mlp.load_state_dict(best_state)
mlp.period = DECISION_PERIOD
mlp.hard_decisions = True
weekly_learned = hard_eval(mlp)
mlp.period = DEPLOY_PERIOD
learned = hard_eval(mlp)
print("Weekly training grid, then the same ω every day.")
summarize_economics("learned (weekly train)", weekly_learned)
summarize_economics("learned (daily deploy)", learned)
summarize_economics("AIMCRA (reference)", standard_practice)
summarize_economics("rainfed", frame=rainfed_df, amounts_cm=zero_amounts)
summarize_economics("potential production", frame=pp_df, potential=True)


## 9. Inspecting the learned control trajectory

The learned controller was trained using weekly decisions, but the resulting weights are then evaluated daily. Importantly, the controller does not replay the weekly irrigation dates. Each day it recomputes its action from the current state.

The resulting trajectory below lets us see the complete control loop: depletion develops, the policy applies water, soil moisture increases, and the subsequent trajectory changes.


In [ ]:
#@title Plot learned closed-loop irrigation trajectory { display-mode: "form" }
smfc = float(provider["SMFCF"])


def add_depletion(frame):
    out = frame.copy()
    out["depletion_mm"] = (smfc - out["SM"]).clip(lower=0.0) * out["RD"] * 10.0
    return out


def rain_series(index, weather_provider):
    vals = [rain_mm_on(weather_provider, day) for day in index]
    return pd.Series(vals, index=index, name="rain_mm")


learned_frame = add_depletion(learned["frame"])
aimcra_frame = add_depletion(standard_practice["frame"])
rainfed_dep = add_depletion(rainfed_df)
rain_mm = rain_series(learned_frame.index, weather)

fig, axes = plt.subplots(4, 1, figsize=(9, 10), sharex=True)

axes[0].plot(rainfed_dep.index, rainfed_dep["depletion_mm"], color="0.6", linewidth=0.9, label="rainfed")
axes[0].plot(learned_frame.index, learned_frame["depletion_mm"], color="C0", label="learned policy")
axes[0].set_ylabel("depletion (mm)")
axes[0].legend(loc="upper left", fontsize=8)

axes[1].bar(rain_mm.index, rain_mm, width=1.0, color="C0", alpha=0.35, label="rain")
axes[1].bar(
    learned_frame.index,
    learned_frame["irrigation"],
    width=1.0,
    color="C1",
    alpha=0.9,
    label="irrigation",
)
axes[1].set_ylabel("mm day$^{-1}$")
axes[1].legend(loc="upper right", fontsize=8)

axes[2].plot(learned_frame.index, learned_frame["DVS"], color="C0")
axes[2].set_ylabel("DVS")

axes[3].plot(rainfed_df.index, rainfed_df["TWSO"], color="0.6", label="rainfed")
axes[3].plot(learned_frame.index, learned_frame["TWSO"], color="C0", label="learned")
axes[3].plot(pp_df.index, pp_df["TWSO"], color="C2", linestyle=":", label="potential")
axes[3].set_ylabel("TWSO (kg ha$^{-1}$)")
axes[3].set_xlabel("day")
axes[3].legend(loc="upper left", fontsize=8)

fig.suptitle("Closed-loop irrigation on the 2010 training year", y=0.99)
fig.tight_layout()
plt.show()


## 10. What policy has been learned?

A calendar would be a list of dates. This is a map from state to action.

For example, two days with the same date in different years do not necessarily produce the same action. What matters is the state encountered on that day: how depleted the root zone is, whether rain is occurring, and where the crop is in its development.

On each day the net outputs $p(\mathrm{irrigate})$ from depletion, rain, DVS and day of year. The scatter below is that map under daily deployment on the training season: $p$ against root-zone depletion, with hard ON days marked.


In [ ]:
#@title Plot policy map: p vs root-zone depletion { display-mode: "form" }
rows = []
for rec in mlp.records:
    rows.append(
        {
            "p": float(rec["p"]),
            "depletion_mm": float(rec["depletion_mm"]),
            "dvs": float(rec["dvs"]),
            "on": float(rec["applied_mm"]) >= EVENT_MM_MIN,
        }
    )
decisions = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
off = decisions[~decisions["on"]]
on = decisions[decisions["on"]]
sc = ax.scatter(
    off["depletion_mm"],
    off["p"],
    c=off["dvs"],
    cmap="viridis",
    vmin=0,
    vmax=2,
    marker="o",
    s=36,
    alpha=0.75,
    label="OFF",
)
ax.scatter(
    on["depletion_mm"],
    on["p"],
    c=on["dvs"],
    cmap="viridis",
    vmin=0,
    vmax=2,
    marker="x",
    s=70,
    linewidths=2,
    label="ON ($p>0.5$)",
)
ax.axhline(0.5, color="0.4", linestyle=":", linewidth=0.8)
ax.set_xlabel("root-zone depletion (mm)")
ax.set_ylabel("$p(\\mathrm{irrigate})$")
ax.set_ylim(-0.05, 1.05)
cb = fig.colorbar(sc, ax=ax)
cb.set_label("DVS")
ax.legend(loc="lower right")
ax.set_title("Learned map: depletion, DVS $\\rightarrow$ irrigate")
fig.tight_layout()
plt.show()


## 11. Does the policy generalize to different weather?

The policy was fit on **one** rainfall series (YAML 2010). Sowing stays 27 March and harvest 31 December; only the campaign year changes. Test weather is NASA POWER 2009 and 2011 at the same coordinates.

We now freeze the policy and change only the weather year.

**What would a memorized calendar do?**

Suppose the network had effectively learned:

“Irrigate on these twelve dates.”

Then changing the weather year would not change the dates on which it irrigates.

That is not what we should expect from a closed-loop policy. A state-dependent controller should instead respond to the different soil-water and crop trajectories produced by the new weather.

AIMCRA is re-evaluated on each year (it has no trained weights). The MLP is frozen at the 2010 checkpoint and deployed every day.


In [ ]:
#@title Freeze 2010 weights; evaluate 2009 and 2011 { display-mode: "form" }
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if hasattr(obj, "to_string") else obj)


nasa_weather = NASAPowerWindow(
    SITE_LAT,
    SITE_LON,
    start=dt.date(2008, 1, 1),
    end=dt.date(2012, 12, 31),
)
print(
    f"NASA POWER {SITE_LAT:.5f}°N, {abs(SITE_LON):.5f}°W  "
    f"{nasa_weather.first_date} – {nasa_weather.last_date}"
)

mlp_frozen = IrrigationMLP(period=DEPLOY_PERIOD)
mlp_frozen.load_state_dict(clone_state(mlp))
mlp_frozen.hard_decisions = True


def crop_window(agro):
    spec = next(iter(agro[0].values()))
    calendar = spec["CropCalendar"]
    return calendar["crop_start_date"], calendar["crop_end_date"]


def season_rain_mm(weather_provider, start, end):
    total = 0.0
    day = start
    one = dt.timedelta(days=1)
    while day <= end:
        total += rain_mm_on(weather_provider, day)
        day += one
    return total


def comparison_row(twso, amounts_cm, potential=False):
    y = scalarize(fresh_yield_t_ha(twso))
    if potential:
        return {
            "fresh yield (t/ha)": y,
            "applied irrigation (mm)": float("nan"),
            "events": float("nan"),
            "reward (€/ha)": float("nan"),
        }
    eco = irrigation_economics(
        torch.as_tensor(scalarize(twso), dtype=ComputeConfig.get_dtype()), amounts_cm
    )
    return {
        "fresh yield (t/ha)": y,
        "applied irrigation (mm)": scalarize(eco["applied_mm"]),
        "events": eco["n_events_hard"],
        "reward (€/ha)": scalarize(eco["reward"]),
    }


def as_engine_weather(weather_provider, agro):
    if weather_is_pcse_provider(weather_provider) and isinstance(weather, list):
        start, end = crop_window(agro)
        return pcse_provider_to_engine_weather(weather_provider, start, end)
    return weather_provider
    engine = Engine(config=pp_config)
    engine.setup(provider, weather_provider, agro)
    engine.run_till_terminate()
    results = engine.get_output()
    return {"twso": results[-1]["TWSO"], "amounts_cm": None}


def evaluate_season(season, weather_provider, agro, reuse=None):
    start, end = crop_window(agro)
    rain_mm = season_rain_mm(weather_provider, start, end)
    print(f"\n{season}: season rain {rain_mm:.0f} mm")
    records = []
    specs = [
        ("rainfed", None, False),
        ("AIMCRA", StandardPracticePolicy(), False),
        ("learned policy (2010)", mlp_frozen, False),
        ("potential production", None, True),
    ]
    for name, ctrl, potential in specs:
        w_engine = as_engine_weather(weather_provider, agro)
        if reuse is not None and name in reuse:
            outcome = reuse[name]
        elif potential:
            engine = Engine(config=pp_config)
            engine.setup(provider, w_engine, agro)
            engine.run_till_terminate()
            results = engine.get_output()
            outcome = {"twso": results[-1]["TWSO"], "amounts_cm": None}
        elif ctrl is None:
            with torch.no_grad():
                outcome = evaluate_policy(None, provider, w_engine, agro, wlp_config)
        else:
            with torch.no_grad():
                outcome = evaluate_policy(ctrl, provider, w_engine, agro, wlp_config)
        rec = comparison_row(outcome["twso"], outcome.get("amounts_cm"), potential=potential)
        rec["season"] = season
        rec["season rain (mm)"] = rain_mm
        rec["controller"] = name
        records.append(rec)
        if potential:
            print(f"  {name:24s}  fresh={rec['fresh yield (t/ha)']:5.1f} t/ha  (yield ceiling)")
        else:
            print(
                f"  {name:24s}  fresh={rec['fresh yield (t/ha)']:5.1f} t/ha  "
                f"applied={rec['applied irrigation (mm)']:5.0f} mm  "
                f"events={int(rec['events']):3d}  "
                f"R={rec['reward (€/ha)']:7.0f} €/ha"
            )
    return records


transfer_rows = []
transfer_rows.extend(
    evaluate_season(
        f"{TRAIN_YEAR} YAML (train)",
        weather,
        agromanagement,
        reuse={
            "rainfed": {"twso": rainfed_df["TWSO"].iloc[-1], "amounts_cm": zero_amounts},
            "AIMCRA": standard_practice,
            "learned policy (2010)": learned,
            "potential production": {"twso": pp_df["TWSO"].iloc[-1], "amounts_cm": None},
        },
    )
)
for year in TEST_YEARS:
    transfer_rows.extend(
        evaluate_season(
            f"{year} NASA POWER (test)", nasa_weather, agro_for_year(yaml_agro, year)
        )
    )

transfer = pd.DataFrame(transfer_rows)
controller_order = ["rainfed", "AIMCRA", "learned policy (2010)"]
season_order = [f"{TRAIN_YEAR} YAML (train)"] + [
    f"{year} NASA POWER (test)" for year in TEST_YEARS
]
reward_table = transfer.pivot(index="controller", columns="season", values="reward (€/ha)")
reward_table = reward_table.reindex(
    index=controller_order,
    columns=[name for name in season_order if name in reward_table.columns],
)
print("\nReward (€/ha) by season")
display(reward_table.round(0))

ax = reward_table.loc[["rainfed", "AIMCRA", "learned policy (2010)"]].T.plot(
    kind="bar", figsize=(8.5, 4.0), rot=15
)
ax.set_ylabel("reward (€ ha$^{-1}$)")
ax.set_xlabel("")
ax.legend(fontsize=8, loc="upper right")
ax.set_title("Frozen 2010 policy on held-out weather years")
fig = ax.get_figure()
fig.tight_layout()
plt.show()


**What happens?** The same frozen network produces different irrigation events in 2009 and 2011 — and different yields and returns — even though the weights never change.

Why? Because the network does not see the date alone. It sees the current state of the crop and soil. Different weather produces different soil-water trajectories, which produce different observations, which produce different actions.

This is the defining property of the closed-loop policy demonstrated here: a function of state, not a memorised schedule.


## 12. Limitations and take-away

- Training uses a single weather year; a stricter protocol would fit $\omega$ on several seasons.
- The action is a simplified event (fixed 50 mm gift, STE hard gate), trained weekly and deployed daily, not a farm implement or a continuous pump.
- STE is a biased surrogate for the discrete decision; the reported policy is the best *hard* $R$, not the last Adam step.
- These are simulation results on one YAML soil, not field validation.

The point of the example is the graph: a process-based crop model can be a control environment. A neural policy observes crop and soil state and chooses management. Because the model is differentiable, the consequences of those actions propagate back to $\omega$, and the policy can be optimized with standard gradient methods.
